In [ ]:
#HOUDEN!!!

import numpy as np
from mcap.reader import make_reader
from mcap_ros2.decoder import Decoder
import struct

def extract_merged_lidar_frame(mcap_path, output_bin, target_timestamp_float, lidar_topics):
    target_ns = int(target_timestamp_float * 1e9)
    # We zoeken nu in een ruimer venster van 0.5 seconden rond de tijd
    start_search = target_ns - 250_000_000 
    end_search = target_ns + 250_000_000

    all_points_list = []

    with open(mcap_path, "rb") as f:
        reader = make_reader(f)
        decoder = Decoder()
        
        for topic in lidar_topics:
            print(f"Zoeken naar dichtstbijzijnde frame voor {topic}...")
            best_msg = None
            min_diff = float('inf')
            best_schema = None

            # Stap 1: Zoek het bericht dat qua tijd het dichtst bij target_ns ligt
            for schema, channel, message in reader.iter_messages(
                topics=[topic], 
                start_time=start_search, 
                end_time=end_search
            ):
                diff = abs(message.log_time - target_ns)
                if diff < min_diff:
                    min_diff = diff
                    best_msg = message
                    best_schema = schema

            if best_msg:
                # Stap 2: Decodeer en extraheer het beste bericht
                try:
                    ros_msg = decoder.decode(best_schema, best_msg)
                except:
                    ros_msg = decoder.decode(best_msg)
                
                print(f"-> Gevonden op {best_msg.log_time / 1e9} (verschil: {min_diff/1e6:.1f}ms)")
                
                point_step = ros_msg.point_step
                num_points = ros_msg.width * ros_msg.height
                data = ros_msg.data
                topic_points = np.zeros((num_points, 5), dtype=np.float32)
                
                # Betrouwbare extractie
                for i in range(num_points):
                    offset = i * point_step
                    try:
                        # x, y, z, intensity (floats)
                        pt = struct.unpack_from('ffff', data, offset)
                        topic_points[i, 0:4] = pt
                        # Ring index (proberen op offset 16)
                        ring = struct.unpack_from('H', data, offset + 16)[0]
                        topic_points[i, 4] = float(ring)
                    except:
                        continue
                
                all_points_list.append(topic_points)
                found_topic = True
            else:
                print(f"-> WAARSCHUWING: Helemaal geen data gevonden voor {topic}")

    if all_points_list:
        final_merged_points = np.vstack(all_points_list)
        final_merged_points.tofile(output_bin)
        print(f"\nSucces! Totaal {len(final_merged_points)} punten opgeslagen in:\n{output_bin}")
    else:
        print("Geen data gevonden.")

# --- INSTELLINGEN ---
MCAP_FILE = r"C:\rosbag_0.mcap"
OUTPUT_FILE = r"C:\Users\Lars Wissink\OneDrive\Documenten\lars wissink\WB TU Delft\jaar 3\bep\.mcap to Bin downloads\merged_full_scan.bin"
TARGET_TIME = 1762960978.527071919

TOPICS_TO_MERGE = [
    "/rslidar/M1P_deskewed",
    "/rslidar/helios_L",
    "/rslidar/helios_R"
]

extract_merged_lidar_frame(MCAP_FILE, OUTPUT_FILE, TARGET_TIME, TOPICS_TO_MERGE)

Zoeken naar dichtstbijzijnde frame voor /rslidar/M1P_deskewed...
-> Gevonden op 1762960978.527072 (verschil: 0.0ms)
Zoeken naar dichtstbijzijnde frame voor /rslidar/helios_L...
-> Gevonden op 1762960978.5111113 (verschil: 16.0ms)
Zoeken naar dichtstbijzijnde frame voor /rslidar/helios_R...
-> Gevonden op 1762960978.5119395 (verschil: 15.1ms)

Succes! Totaal 193310 punten opgeslagen in:
C:\Users\Lars Wissink\OneDrive\Documenten\lars wissink\WB TU Delft\jaar 3\bep\.mcap to Bin downloads\merged_full_scan.bin


In [5]:
#houden!!!
import open3d as o3d
import numpy as np
import time

bin_path = r"C:\Users\Lars Wissink\OneDrive\Documenten\lars wissink\WB TU Delft\jaar 3\bep\.mcap to Bin downloads\trajectory_map.bin"
print("1. Bestand laden...")
data = np.fromfile(bin_path, dtype=np.float32).reshape(-1, 5)
initial_count = len(data)

# STAP 1: Verwijder alle rijen die NaN bevatten in de eerste 3 kolommen (XYZ)
print("2. Ongeldige punten (NaN) verwijderen...")
mask = ~np.isnan(data[:, :3]).any(axis=1)
clean_data = data[mask]

# STAP 2: Verwijder punten die op (0,0,0) staan (vaak ook ruis)
mask_zero = ~((clean_data[:, 0] == 0) & (clean_data[:, 1] == 0) & (clean_data[:, 2] == 0))
clean_data = clean_data[mask_zero]

final_count = len(clean_data)
print(f"   -> Gefilterd: {initial_count - final_count} punten verwijderd.")
print(f"   -> Overgebleven: {final_count} punten.")

# STAP 3: Omzetten naar Open3D
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(clean_data[:, :3])

# Kleuren op basis van intensiteit (kolom 4)
intensities = clean_data[:, 3]
# Normaliseer intensiteit voor kleuren (0 tot 1)
if intensities.max() > 0:
    colors = intensities / intensities.max()
    # Maak een grijswaarden-map (RGB waarbij R=G=B)
    rgb_colors = np.stack([colors, colors, colors], axis=1)
    pcd.colors = o3d.utility.Vector3dVector(rgb_colors)

# STAP 4: Downsampling (nu zonder hangen!)
print("3. Voxel downsampling (0.1m)...")
pcd = pcd.voxel_down_sample(voxel_size=0.1)

# STAP 5: Tonen
print("4. Visualisatie openen...")
o3d.visualization.draw_geometries([pcd], 
                                  window_name=f"Cleaned Cloud - {final_count} pts",
                                  width=1200, height=800)

1. Bestand laden...
2. Ongeldige punten (NaN) verwijderen...
   -> Gefilterd: 0 punten verwijderd.
   -> Overgebleven: 2458448 punten.
3. Voxel downsampling (0.1m)...
4. Visualisatie openen...


In [1]:
import numpy as np
from mcap.reader import make_reader
from mcap_ros2.decoder import Decoder
import struct
from scipy.spatial.transform import Rotation as R

def get_transformation_matrix(odom_msg):
    """Zet een Odometry bericht om in een 4x4 matrix."""
    pos = odom_msg.pose.pose.position
    ori = odom_msg.pose.pose.orientation
    rotation = R.from_quat([ori.x, ori.y, ori.z, ori.w]).as_matrix()
    T = np.eye(4)
    T[:3, :3] = rotation
    T[:3, 3] = [pos.x, pos.y, pos.z]
    return T

def extract_odometry_corrected_lidar(mcap_path, output_bin, start_time, end_time, step_seconds, lidar_topics, odom_topic):
    start_ns = int(start_time * 1e9)
    end_ns = int(end_time * 1e9)
    step_ns = int(step_seconds * 1e9)

    all_points_list = []
    first_odom_matrix_inv = None  # Om alles naar (0,0,0) te trekken

    with open(mcap_path, "rb") as f:
        reader = make_reader(f)
        decoder = Decoder()
        
        target_timestamps = range(start_ns, end_ns, step_ns)

        for target_ns in target_timestamps:
            print(f"\n--- Verwerken: {target_ns / 1e9} ---")
            
            # 1. Zoek dichtstbijzijnde Odometrie
            best_odom_msg = None
            min_odom_diff = float('inf')
            # Ruimer venster voor odom omdat dit vaak op lagere frequentie draait
            for schema, channel, message in reader.iter_messages(topics=[odom_topic], start_time=target_ns-200_000_000, end_time=target_ns+200_000_000):
                diff = abs(message.log_time - target_ns)
                if diff < min_odom_diff:
                    min_odom_diff = diff
                    best_odom_msg = decoder.decode(schema, message)

            if not best_odom_msg:
                print(f"  [!] Geen odom gevonden voor {target_ns/1e9}, overslaan...")
                continue

            # Bereken de huidige wereld-matrix
            T_world_robot = get_transformation_matrix(best_odom_msg)

            # Als dit het eerste frame is, bepaal dan de oorsprong
            if first_odom_matrix_inv is None:
                print("  [INFO] Eerste frame gevonden. Dit wordt het nulpunt (0,0,0).")
                first_odom_matrix_inv = np.linalg.inv(T_world_robot)

            # Bereken de relatieve matrix: Verplaatsing t.o.v. startpunt
            T_relative = first_odom_matrix_inv @ T_world_robot

            # 2. Haal LiDAR data van dit tijdstip
            for topic in lidar_topics:
                best_lidar_msg = None
                min_lidar_diff = float('inf')
                best_schema = None

                for schema, channel, message in reader.iter_messages(topics=[topic], start_time=target_ns-100_000_000, end_time=target_ns+100_000_000):
                    diff = abs(message.log_time - target_ns)
                    if diff < min_lidar_diff:
                        min_lidar_diff = diff
                        best_lidar_msg = message
                        best_schema = schema

                if best_lidar_msg:
                    ros_msg = decoder.decode(best_schema, best_lidar_msg)
                    
                    # Punten extraheren
                    point_step = ros_msg.point_step
                    num_points = ros_msg.width * ros_msg.height
                    data = ros_msg.data
                    
                    # Buffer voor transformatie
                    local_xyz = []
                    other_data = [] # intensity en ring

                    for i in range(num_points):
                        offset = i * point_step
                        try:
                            pt = struct.unpack_from('ffff', data, offset)
                            if not np.isnan(pt[0]): # Filter direct NaNs
                                local_xyz.append([pt[0], pt[1], pt[2]])
                                ring = struct.unpack_from('H', data, offset + 16)[0]
                                other_data.append([pt[3], float(ring)])
                        except: continue
                    
                    if not local_xyz: continue

                    # Transformeren naar relatieve coördinaten
                    pts_local = np.array(local_xyz)
                    ones = np.ones((pts_local.shape[0], 1))
                    pts_hom = np.hstack([pts_local, ones])
                    
                    # P_relatief = T_relative * P_local
                    pts_rel = (T_relative @ pts_hom.T).T[:, :3]
                    
                    # Samenvoegen met intensity en ring
                    combined = np.hstack([pts_rel, np.array(other_data)])
                    all_points_list.append(combined.astype(np.float32))
                    print(f"  [OK] {topic} toegevoegd en getransformeerd.")

    if all_points_list:
        final_cloud = np.vstack(all_points_list)
        final_cloud.tofile(output_bin)
        print(f"\nSucces! Bestand opgeslagen: {output_bin}")
    else:
        print("Geen data kunnen verwerken.")

# --- INSTELLINGEN ---
MCAP_FILE = r"C:\rosbag_0.mcap"
OUTPUT_FILE = r"C:\Users\Lars Wissink\OneDrive\Documenten\lars wissink\WB TU Delft\jaar 3\bep\.mcap to Bin downloads\trajectory_map.bin"

START = 1762960978.580190803
END = 1762960986.981583724
INTERVAL = 0.5  # 0.5 seconde geeft een mooie dichtheid zonder dat het te traag wordt

TOPICS = ["/rslidar/M1P_deskewed", "/rslidar/helios_L", "/rslidar/helios_R"]
ODOM_TOPIC = "/odometry/global"

extract_odometry_corrected_lidar(MCAP_FILE, OUTPUT_FILE, START, END, INTERVAL, TOPICS, ODOM_TOPIC)


--- Verwerken: 1762960978.580191 ---
  [INFO] Eerste frame gevonden. Dit wordt het nulpunt (0,0,0).
  [OK] /rslidar/M1P_deskewed toegevoegd en getransformeerd.
  [OK] /rslidar/helios_L toegevoegd en getransformeerd.
  [OK] /rslidar/helios_R toegevoegd en getransformeerd.

--- Verwerken: 1762960979.080191 ---
  [OK] /rslidar/M1P_deskewed toegevoegd en getransformeerd.
  [OK] /rslidar/helios_L toegevoegd en getransformeerd.
  [OK] /rslidar/helios_R toegevoegd en getransformeerd.

--- Verwerken: 1762960979.580191 ---
  [OK] /rslidar/M1P_deskewed toegevoegd en getransformeerd.
  [OK] /rslidar/helios_L toegevoegd en getransformeerd.
  [OK] /rslidar/helios_R toegevoegd en getransformeerd.

--- Verwerken: 1762960980.080191 ---
  [OK] /rslidar/M1P_deskewed toegevoegd en getransformeerd.
  [OK] /rslidar/helios_L toegevoegd en getransformeerd.
  [OK] /rslidar/helios_R toegevoegd en getransformeerd.

--- Verwerken: 1762960980.580191 ---
  [OK] /rslidar/M1P_deskewed toegevoegd en getransformeerd.